# AEZ-26 Phase 2F — QC-Qualified Native-Time Response Metrics and Lag Analysis

This notebook converts the **Phase 2E native response links** into event-level response metrics while keeping all response datasets at their native temporal support.

It automatically:

- creates `Phase2/Phase2F/`;
- derives seasonal standardized response anomalies;
- calculates pre-event baseline, event-period anomaly, response minimum, lag and observed recovery;
- retains MOD13 QA sensitivities;
- flags low-frequency event confounding;
- creates a 39-event cross-response table;
- saves tables, figures and one ZIP audit package.

**Flash-drought labels remain frozen.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

BASE_DIR = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE2_DIR = BASE_DIR / 'Phase2'
PHASE2E_DIR = PHASE2_DIR / 'Phase2E'
PHASE2F_DIR = PHASE2_DIR / 'Phase2F'
FIG_DIR = PHASE2F_DIR / 'Figures'

PHASE2F_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Phase 2E links
MOD16_LINK_FILE = PHASE2E_DIR / 'AEZ26_Phase2E_MOD16_Native_Response_Links.csv'
MOD13_LINK_FILE = PHASE2E_DIR / 'AEZ26_Phase2E_MOD13_Native_Response_Links.csv'
GRACE_LINK_FILE = PHASE2E_DIR / 'AEZ26_Phase2E_GRACE_TWSA_Native_Response_Links.csv'
JRC_LINK_FILE = PHASE2E_DIR / 'AEZ26_Phase2E_JRC_Monthly_Response_Links.csv'
AVAIL_FILE = PHASE2E_DIR / 'AEZ26_Phase2E_Event_Response_Availability_Summary.csv'

# Original response sources for climatologies
MOD16_SOURCE = BASE_DIR / 'AEZ26_MODIS_ET_PET_QC_2000_2025.csv'
MOD13_SOURCE = BASE_DIR / 'AEZ26_MODIS_NDVI_EVI_QC_2000_2025.csv'
GRACE_SOURCE = BASE_DIR / 'AEZ26_GRACE_Mascon_TWSA_Uncertainty.csv'
JRC_SOURCE = BASE_DIR / 'AEZ26_JRC_SurfaceWater_Monthly_QC_2000_2021.csv'

required = [
    MOD16_LINK_FILE, MOD13_LINK_FILE, GRACE_LINK_FILE,
    JRC_LINK_FILE, AVAIL_FILE,
    MOD16_SOURCE, MOD13_SOURCE, GRACE_SOURCE, JRC_SOURCE
]

for p in required:
    print(p.name, '->', p.exists())
    if not p.exists():
        raise FileNotFoundError(f'Missing required input: {p}')

print('Phase 2F output folder:', PHASE2F_DIR)


In [ ]:
mod16_link = pd.read_csv(
    MOD16_LINK_FILE,
    parse_dates=[
        'onset_start_date','threshold_crossing_end_date','event_end_date',
        'native_start_date','native_end_date','native_mid_date',
        'context_window_start','context_window_end'
    ]
)

mod13_link = pd.read_csv(
    MOD13_LINK_FILE,
    parse_dates=[
        'onset_start_date','threshold_crossing_end_date','event_end_date',
        'native_composite_start_date','context_window_start','context_window_end'
    ]
)

grace_link = pd.read_csv(
    GRACE_LINK_FILE,
    parse_dates=[
        'onset_start_date','threshold_crossing_end_date','event_end_date',
        'native_start_date','native_end_date','native_mid_date',
        'context_window_start','context_window_end'
    ]
)

jrc_link = pd.read_csv(
    JRC_LINK_FILE,
    parse_dates=[
        'onset_start_date','threshold_crossing_end_date','event_end_date',
        'native_month_start','native_month_end','native_month_mid',
        'context_window_start','context_window_end'
    ]
)

availability = pd.read_csv(
    AVAIL_FILE,
    parse_dates=['onset_start_date','event_end_date']
)

mod16 = pd.read_csv(
    MOD16_SOURCE,
    parse_dates=['composite_start_date','composite_end_date']
)

mod13 = pd.read_csv(
    MOD13_SOURCE,
    parse_dates=['composite_start_date']
)

grace = pd.read_csv(
    GRACE_SOURCE,
    parse_dates=['native_date','native_end_date_if_available']
)

jrc = pd.read_csv(
    JRC_SOURCE,
    parse_dates=['date']
)

assert len(availability) == 39
assert availability['event_id'].is_unique

assert len(mod16) == 1196
assert len(mod13) == 595
assert len(grace) == 238
assert len(jrc) == 264

print('Input gate: PASS')


## 1. Helper functions


In [ ]:
def group_zscore(df, group_col, value_col):
    mean = df.groupby(group_col)[value_col].transform('mean')
    sd = df.groupby(group_col)[value_col].transform('std')
    count = df.groupby(group_col)[value_col].transform('count')

    z = (df[value_col] - mean) / sd

    return z, mean, sd, count


def empirical_percentile(value, reference):
    ref = np.asarray(reference, dtype=float)
    ref = ref[np.isfinite(ref)]

    if not np.isfinite(value) or len(ref) == 0:
        return np.nan

    less = np.sum(ref < value)
    equal = np.sum(ref == value)

    return 100.0 * (
        less + (equal + 1.0) / 2.0
    ) / len(ref)


def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    ok = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    if not ok.any():
        return np.nan

    return float(np.average(values[ok], weights=weights[ok]))


def overlap_days(a_start, a_end, b_start, b_end):
    start = max(pd.Timestamp(a_start), pd.Timestamp(b_start))
    end = min(pd.Timestamp(a_end), pd.Timestamp(b_end))

    if end < start:
        return 0

    return int((end - start).days + 1)


def recovery_dates(
    df,
    date_col,
    z_col,
    minimum_date,
    context_end,
    pre_baseline
):
    later = df.loc[
        (df[date_col] > minimum_date)
        &
        (df[date_col] <= context_end)
        &
        df[z_col].notna()
    ].sort_values(date_col)

    normal = later.loc[later[z_col] >= 0]
    baseline = later.loc[
        later[z_col] >= pre_baseline
    ] if np.isfinite(pre_baseline) else later.iloc[0:0]

    normal_date = (
        normal.iloc[0][date_col]
        if len(normal) else pd.NaT
    )

    baseline_date = (
        baseline.iloc[0][date_col]
        if len(baseline) else pd.NaT
    )

    return normal_date, baseline_date


## 2. Build native seasonal climatologies


In [ ]:
# -------------------------
# MOD16 — 8-day slots
# -------------------------
mod16['seasonal_slot_8day'] = (
    ((mod16['composite_start_date'].dt.dayofyear - 1) // 8) + 1
).astype(int)

for variable in [
    'ET_PET_ratio_primary',
    'ET_primary_mm_day',
    'PET_primary_mm_day',
    'ET_PET_ratio_strictQA'
]:
    z, mean, sd, count = group_zscore(
        mod16,
        'seasonal_slot_8day',
        variable
    )

    mod16[f'{variable}_z'] = z
    mod16[f'{variable}_clim_mean'] = mean
    mod16[f'{variable}_clim_sd'] = sd
    mod16[f'{variable}_clim_n'] = count

assert mod16.groupby('seasonal_slot_8day').size().min() == 26
assert mod16.groupby('seasonal_slot_8day').size().max() == 26


# -------------------------
# MOD13 — 16-day slots
# -------------------------
mod13['seasonal_slot_16day'] = (
    ((mod13['composite_start_date'].dt.dayofyear - 1) // 16) + 1
).astype(int)

for variable in [
    'NDVI_standardQA',
    'EVI_standardQA',
    'NDVI_strictQA',
    'EVI_strictQA'
]:
    z, mean, sd, count = group_zscore(
        mod13,
        'seasonal_slot_16day',
        variable
    )

    mod13[f'{variable}_z'] = z
    mod13[f'{variable}_clim_mean'] = mean
    mod13[f'{variable}_clim_sd'] = sd
    mod13[f'{variable}_clim_n'] = count


# -------------------------
# GRACE — calendar month
# -------------------------
grace['calendar_month'] = grace['native_date'].dt.month

z, mean, sd, count = group_zscore(
    grace,
    'calendar_month',
    'TWSA_cm'
)

grace['TWSA_z'] = z
grace['TWSA_monthly_clim_mean_cm'] = mean
grace['TWSA_monthly_clim_sd_cm'] = sd
grace['TWSA_monthly_clim_n'] = count
grace['TWSA_deseasonalized_cm'] = (
    grace['TWSA_cm']
    - grace['TWSA_monthly_clim_mean_cm']
)


# -------------------------
# JRC — qualified monthly climatology
# -------------------------
jrc['calendar_month'] = jrc['date'].dt.month

jrc['coverage_qualified_flag'] = (
    (jrc['no_valid_observation_flag'] == 0)
    &
    (jrc['low_coverage_flag'] == 0)
).astype(int)

jrc_qualified = jrc.loc[
    jrc['coverage_qualified_flag'] == 1
].copy()

jrc_month_stats = (
    jrc_qualified
    .groupby('calendar_month')[
        'water_fraction_of_valid_area'
    ]
    .agg(['mean','std','median','count'])
    .rename(columns={
        'mean':'JRC_clim_mean',
        'std':'JRC_clim_sd',
        'median':'JRC_clim_median',
        'count':'JRC_clim_n'
    })
)

jrc = jrc.merge(
    jrc_month_stats,
    left_on='calendar_month',
    right_index=True,
    how='left'
)

jrc['water_fraction_z'] = np.where(
    (jrc['coverage_qualified_flag'] == 1)
    &
    (jrc['JRC_clim_n'] >= 5)
    &
    (jrc['JRC_clim_sd'] > 0),
    (
        jrc['water_fraction_of_valid_area']
        - jrc['JRC_clim_mean']
    )
    / jrc['JRC_clim_sd'],
    np.nan
)

jrc['water_fraction_deviation_from_monthly_median'] = np.where(
    (jrc['coverage_qualified_flag'] == 1)
    &
    (jrc['JRC_clim_n'] >= 5),
    (
        jrc['water_fraction_of_valid_area']
        - jrc['JRC_clim_median']
    ),
    np.nan
)

print('MOD16 seasonal slots:', mod16['seasonal_slot_8day'].nunique())
print('MOD13 seasonal slots:', mod13['seasonal_slot_16day'].nunique())
print('GRACE month reference n range:',
      int(grace['TWSA_monthly_clim_n'].min()),
      'to',
      int(grace['TWSA_monthly_clim_n'].max()))
print('JRC qualified climatology by month:')
display(jrc_month_stats)


## 3. Attach standardized anomalies to the Phase 2E native link tables


In [ ]:
# MOD16
mod16_anomaly_cols = [
    'composite_start_date',
    'seasonal_slot_8day',
    'ET_PET_ratio_primary_z',
    'ET_primary_mm_day_z',
    'PET_primary_mm_day_z',
    'ET_PET_ratio_strictQA_z'
]

mod16_links = mod16_link.merge(
    mod16[mod16_anomaly_cols],
    left_on='native_start_date',
    right_on='composite_start_date',
    how='left',
    validate='many_to_one'
).drop(columns=['composite_start_date'])

assert mod16_links[
    'ET_PET_ratio_primary_z'
].notna().all()


# MOD13
mod13_anomaly_cols = [
    'composite_start_date',
    'seasonal_slot_16day',
    'NDVI_standardQA_z',
    'EVI_standardQA_z',
    'NDVI_strictQA_z',
    'EVI_strictQA_z'
]

mod13_links = mod13_link.merge(
    mod13[mod13_anomaly_cols],
    left_on='native_composite_start_date',
    right_on='composite_start_date',
    how='left',
    validate='many_to_one'
).drop(columns=['composite_start_date'])


# GRACE
grace_anomaly_cols = [
    'native_system_index',
    'calendar_month',
    'TWSA_z',
    'TWSA_deseasonalized_cm',
    'TWSA_monthly_clim_mean_cm',
    'TWSA_monthly_clim_sd_cm',
    'TWSA_monthly_clim_n'
]

grace_links = grace_link.merge(
    grace[grace_anomaly_cols],
    on='native_system_index',
    how='left',
    validate='many_to_one'
)

assert grace_links['TWSA_z'].notna().all()


# JRC
jrc_anomaly_cols = [
    'native_system_index',
    'calendar_month',
    'JRC_clim_mean',
    'JRC_clim_sd',
    'JRC_clim_median',
    'JRC_clim_n',
    'water_fraction_z',
    'water_fraction_deviation_from_monthly_median'
]

jrc_links = jrc_link.merge(
    jrc[jrc_anomaly_cols],
    on='native_system_index',
    how='left',
    validate='many_to_one'
)

print('Attached anomaly tables: PASS')


## 4. MOD16 event-response metrics

Lower ET/PET z = stronger evaporative stress relative to the seasonal climatology.

The metric is MOD16-derived and is **not named ESI**.


In [ ]:
mod16_rows = []

for event_id, g in mod16_links.groupby('event_id'):

    g = g.sort_values('native_mid_date').copy()

    onset = g['onset_start_date'].iloc[0]
    event_end = g['event_end_date'].iloc[0]
    context_end = g['context_window_end'].iloc[0]

    pre_start = onset - pd.Timedelta(days=32)
    pre_end = onset - pd.Timedelta(days=1)

    g['pre_overlap_days'] = [
        overlap_days(
            s, e,
            pre_start, pre_end
        )
        for s, e in zip(
            g['native_start_date'],
            g['native_end_date']
        )
    ]

    pre_primary = weighted_mean(
        g['ET_PET_ratio_primary_z'],
        g['pre_overlap_days']
    )

    event_primary = weighted_mean(
        g['ET_PET_ratio_primary_z'],
        g['overlap_event_days']
    )

    event_et = weighted_mean(
        g['ET_primary_mm_day_z'],
        g['overlap_event_days']
    )

    event_pet = weighted_mean(
        g['PET_primary_mm_day_z'],
        g['overlap_event_days']
    )

    pre_strict = weighted_mean(
        g['ET_PET_ratio_strictQA_z'],
        g['pre_overlap_days']
    )

    event_strict = weighted_mean(
        g['ET_PET_ratio_strictQA_z'],
        g['overlap_event_days']
    )

    candidate = g.loc[
        (g['native_mid_date'] >= onset)
        &
        (g['native_mid_date'] <= context_end)
    ].copy()

    min_idx = candidate[
        'ET_PET_ratio_primary_z'
    ].idxmin()

    min_z = candidate.loc[
        min_idx,
        'ET_PET_ratio_primary_z'
    ]

    min_date = candidate.loc[
        min_idx,
        'native_mid_date'
    ]

    normal_date, baseline_date = recovery_dates(
        g,
        'native_mid_date',
        'ET_PET_ratio_primary_z',
        min_date,
        context_end,
        pre_primary
    )

    strict_candidate = candidate.loc[
        candidate['ET_PET_ratio_strictQA_z'].notna()
    ]

    strict_min = (
        strict_candidate[
            'ET_PET_ratio_strictQA_z'
        ].min()
        if len(strict_candidate)
        else np.nan
    )

    mod16_rows.append({
        'event_id': event_id,
        'primary_driver_class':
            g['primary_driver_class'].iloc[0],
        'onset_season':
            g['onset_season'].iloc[0],

        'prebaseline_ETPET_z':
            pre_primary,

        'event_weighted_ETPET_z':
            event_primary,

        'event_weighted_ET_z':
            event_et,

        'event_weighted_PET_z':
            event_pet,

        'minimum_ETPET_z_onset_to_post64':
            min_z,

        'minimum_ETPET_drop_from_prebaseline_z':
            min_z - pre_primary,

        'minimum_ETPET_native_mid_date':
            min_date,

        'lag_onset_to_minimum_ETPET_days':
            int((min_date - onset).days),

        'lag_event_end_to_minimum_ETPET_days':
            int((min_date - event_end).days),

        'recovery_to_seasonal_normal_date':
            normal_date,

        'recovery_to_seasonal_normal_days_after_event_end':
            (
                (normal_date - event_end).days
                if pd.notna(normal_date)
                else np.nan
            ),

        'recovery_to_prebaseline_date':
            baseline_date,

        'recovery_to_prebaseline_days_after_event_end':
            (
                (baseline_date - event_end).days
                if pd.notna(baseline_date)
                else np.nan
            ),

        'seasonal_normal_recovery_censored_flag':
            int(pd.isna(normal_date)),

        'prebaseline_ETPET_z_strictQA':
            pre_strict,

        'event_weighted_ETPET_z_strictQA':
            event_strict,

        'minimum_ETPET_z_strictQA_onset_to_post64':
            strict_min,

        'candidate_native_records_onset_to_post64':
            len(candidate)
    })


mod16_metrics = pd.DataFrame(mod16_rows)
mod16_metrics = mod16_metrics.rename(columns={'recovery_to_seasonal_normal_date': 'MOD16_recovery_to_seasonal_normal_date', 'recovery_to_seasonal_normal_days_after_event_end': 'MOD16_recovery_to_seasonal_normal_days_after_event_end', 'recovery_to_prebaseline_date': 'MOD16_recovery_to_prebaseline_date', 'recovery_to_prebaseline_days_after_event_end': 'MOD16_recovery_to_prebaseline_days_after_event_end', 'seasonal_normal_recovery_censored_flag': 'MOD16_seasonal_normal_recovery_censored_flag'})

assert len(mod16_metrics) == 39
assert mod16_metrics['event_id'].is_unique

display(mod16_metrics.head())


## 5. MOD13 vegetation response and QA sensitivity


In [ ]:
mod13_rows = []

for event_id, g in mod13_links.groupby('event_id'):

    g = g.sort_values(
        'native_composite_start_date'
    ).copy()

    onset = g['onset_start_date'].iloc[0]
    event_end = g['event_end_date'].iloc[0]
    context_end = g['context_window_end'].iloc[0]

    pre = g.loc[
        (g['native_composite_start_date'] < onset)
        &
        g['NDVI_standardQA_z'].notna()
    ]

    event = g.loc[
        (g['native_composite_start_date'] >= onset)
        &
        (g['native_composite_start_date'] <= event_end)
        &
        g['NDVI_standardQA_z'].notna()
    ]

    candidate = g.loc[
        (g['native_composite_start_date'] >= onset)
        &
        (g['native_composite_start_date'] <= context_end)
        &
        g['NDVI_standardQA_z'].notna()
    ].copy()

    pre_ndvi = weighted_mean(
        pre['NDVI_standardQA_z'],
        pre['standard_QA_fraction']
    )

    pre_evi = weighted_mean(
        pre['EVI_standardQA_z'],
        pre['standard_QA_fraction']
    )

    event_ndvi = weighted_mean(
        event['NDVI_standardQA_z'],
        event['standard_QA_fraction']
    )

    event_evi = weighted_mean(
        event['EVI_standardQA_z'],
        event['standard_QA_fraction']
    )

    min_ndvi_idx = candidate[
        'NDVI_standardQA_z'
    ].idxmin()

    min_ndvi = candidate.loc[
        min_ndvi_idx,
        'NDVI_standardQA_z'
    ]

    min_ndvi_date = candidate.loc[
        min_ndvi_idx,
        'native_composite_start_date'
    ]

    min_ndvi_coverage = candidate.loc[
        min_ndvi_idx,
        'standard_QA_fraction'
    ]

    min_evi_idx = candidate[
        'EVI_standardQA_z'
    ].idxmin()

    min_evi = candidate.loc[
        min_evi_idx,
        'EVI_standardQA_z'
    ]

    min_evi_date = candidate.loc[
        min_evi_idx,
        'native_composite_start_date'
    ]

    ndvi25 = candidate.loc[
        candidate['standard_QA_fraction'] >= 0.25,
        'NDVI_standardQA_z'
    ].min()

    ndvi50 = candidate.loc[
        candidate['standard_QA_fraction'] >= 0.50,
        'NDVI_standardQA_z'
    ].min()

    evi25 = candidate.loc[
        candidate['standard_QA_fraction'] >= 0.25,
        'EVI_standardQA_z'
    ].min()

    evi50 = candidate.loc[
        candidate['standard_QA_fraction'] >= 0.50,
        'EVI_standardQA_z'
    ].min()

    normal_date, baseline_date = recovery_dates(
        g,
        'native_composite_start_date',
        'NDVI_standardQA_z',
        min_ndvi_date,
        context_end,
        pre_ndvi
    )

    strict_candidate = g.loc[
        (g['native_composite_start_date'] >= onset)
        &
        (g['native_composite_start_date'] <= context_end)
        &
        g['NDVI_strictQA_z'].notna()
    ]

    strict_min_ndvi = (
        strict_candidate['NDVI_strictQA_z'].min()
        if len(strict_candidate)
        else np.nan
    )

    mod13_rows.append({
        'event_id': event_id,
        'primary_driver_class':
            g['primary_driver_class'].iloc[0],
        'onset_season':
            g['onset_season'].iloc[0],

        'prebaseline_NDVI_z':
            pre_ndvi,

        'prebaseline_EVI_z':
            pre_evi,

        'event_weighted_NDVI_z':
            event_ndvi,

        'event_weighted_EVI_z':
            event_evi,

        'minimum_NDVI_z_onset_to_post96':
            min_ndvi,

        'minimum_NDVI_drop_from_prebaseline_z':
            min_ndvi - pre_ndvi,

        'minimum_NDVI_native_anchor_date':
            min_ndvi_date,

        'minimum_NDVI_standard_QA_fraction':
            min_ndvi_coverage,

        'lag_onset_to_minimum_NDVI_days':
            int((min_ndvi_date - onset).days),

        'lag_event_end_to_minimum_NDVI_days':
            int((min_ndvi_date - event_end).days),

        'minimum_EVI_z_onset_to_post96':
            min_evi,

        'minimum_EVI_drop_from_prebaseline_z':
            min_evi - pre_evi,

        'minimum_EVI_native_anchor_date':
            min_evi_date,

        'minimum_NDVI_z_QAfrac_ge_025':
            ndvi25,

        'minimum_NDVI_z_QAfrac_ge_050':
            ndvi50,

        'minimum_EVI_z_QAfrac_ge_025':
            evi25,

        'minimum_EVI_z_QAfrac_ge_050':
            evi50,

        'minimum_NDVI_z_strictQA':
            strict_min_ndvi,

        'recovery_to_seasonal_normal_date':
            normal_date,

        'recovery_to_seasonal_normal_days_after_event_end':
            (
                (normal_date - event_end).days
                if pd.notna(normal_date)
                else np.nan
            ),

        'recovery_to_prebaseline_date':
            baseline_date,

        'recovery_to_prebaseline_days_after_event_end':
            (
                (baseline_date - event_end).days
                if pd.notna(baseline_date)
                else np.nan
            ),

        'seasonal_normal_recovery_censored_flag':
            int(pd.isna(normal_date)),

        'candidate_standardQA_records_onset_to_post96':
            len(candidate)
    })


mod13_metrics = pd.DataFrame(mod13_rows)
mod13_metrics = mod13_metrics.rename(columns={'recovery_to_seasonal_normal_date': 'MOD13_recovery_to_seasonal_normal_date', 'recovery_to_seasonal_normal_days_after_event_end': 'MOD13_recovery_to_seasonal_normal_days_after_event_end', 'recovery_to_prebaseline_date': 'MOD13_recovery_to_prebaseline_date', 'recovery_to_prebaseline_days_after_event_end': 'MOD13_recovery_to_prebaseline_days_after_event_end', 'seasonal_normal_recovery_censored_flag': 'MOD13_seasonal_normal_recovery_censored_flag'})

assert len(mod13_metrics) == 39
assert mod13_metrics['event_id'].is_unique

display(mod13_metrics.head())


## 6. GRACE TWSA response and lag

This remains a **TWSA** branch. It is not groundwater.


In [ ]:
grace_rows = []

event_ids = availability['event_id'].tolist()

for event_id in event_ids:

    g = grace_links.loc[
        grace_links['event_id'] == event_id
    ].sort_values('native_mid_date').copy()

    if len(g) == 0:
        grace_rows.append({
            'event_id': event_id,
            'GRACE_response_metric_available_flag': 0
        })
        continue

    onset = g['onset_start_date'].iloc[0]
    event_end = g['event_end_date'].iloc[0]
    context_end = g['context_window_end'].iloc[0]

    pre = g.loc[
        g['native_mid_date'] < onset
    ]

    pre_baseline = (
        float(pre['TWSA_z'].median())
        if len(pre)
        else np.nan
    )

    event_z = weighted_mean(
        g['TWSA_z'],
        g['overlap_event_days']
    )

    candidate = g.loc[
        (g['native_mid_date'] >= onset)
        &
        (g['native_mid_date'] <= context_end)
        &
        g['TWSA_z'].notna()
    ].copy()

    if len(candidate) == 0:
        grace_rows.append({
            'event_id': event_id,
            'GRACE_response_metric_available_flag': 0,
            'prebaseline_TWSA_z': pre_baseline,
            'prebaseline_GRACE_records': len(pre)
        })
        continue

    min_idx = candidate['TWSA_z'].idxmin()

    min_z = candidate.loc[min_idx, 'TWSA_z']
    min_cm = candidate.loc[
        min_idx,
        'TWSA_deseasonalized_cm'
    ]
    min_date = candidate.loc[
        min_idx,
        'native_mid_date'
    ]

    to_min = candidate.loc[
        candidate['native_mid_date'] <= min_date
    ]

    confounded = int(
        to_min[
            'overlaps_other_primary_event_flag'
        ].max()
    )

    normal_date, baseline_date = recovery_dates(
        g,
        'native_mid_date',
        'TWSA_z',
        min_date,
        context_end,
        pre_baseline
    )

    grace_rows.append({
        'event_id': event_id,
        'primary_driver_class':
            g['primary_driver_class'].iloc[0],
        'onset_season':
            g['onset_season'].iloc[0],

        'GRACE_response_metric_available_flag': 1,

        'prebaseline_GRACE_records':
            len(pre),

        'prebaseline_TWSA_z':
            pre_baseline,

        'event_weighted_TWSA_z':
            event_z,

        'minimum_TWSA_z_onset_to_post365':
            min_z,

        'minimum_TWSA_drop_from_prebaseline_z':
            (
                min_z - pre_baseline
                if np.isfinite(pre_baseline)
                else np.nan
            ),

        'minimum_TWSA_deseasonalized_cm':
            min_cm,

        'minimum_TWSA_native_mid_date':
            min_date,

        'lag_onset_to_minimum_TWSA_days':
            int((min_date - onset).days),

        'lag_event_end_to_minimum_TWSA_days':
            int((min_date - event_end).days),

        'other_flash_drought_before_minimum_TWSA_flag':
            confounded,

        'recovery_to_seasonal_normal_date':
            normal_date,

        'recovery_to_seasonal_normal_days_after_event_end':
            (
                (normal_date - event_end).days
                if pd.notna(normal_date)
                else np.nan
            ),

        'recovery_to_prebaseline_date':
            baseline_date,

        'recovery_to_prebaseline_days_after_event_end':
            (
                (baseline_date - event_end).days
                if pd.notna(baseline_date)
                else np.nan
            ),

        'seasonal_normal_recovery_censored_flag':
            int(pd.isna(normal_date)),

        'candidate_GRACE_records_onset_to_post365':
            len(candidate)
    })


grace_metrics = pd.DataFrame(grace_rows)
grace_metrics = grace_metrics.rename(columns={'recovery_to_seasonal_normal_date': 'GRACE_recovery_to_seasonal_normal_date', 'recovery_to_seasonal_normal_days_after_event_end': 'GRACE_recovery_to_seasonal_normal_days_after_event_end', 'recovery_to_prebaseline_date': 'GRACE_recovery_to_prebaseline_date', 'recovery_to_prebaseline_days_after_event_end': 'GRACE_recovery_to_prebaseline_days_after_event_end', 'seasonal_normal_recovery_censored_flag': 'GRACE_seasonal_normal_recovery_censored_flag'})

assert len(grace_metrics) == 39
assert grace_metrics['event_id'].is_unique

display(grace_metrics.head())


## 7. JRC coverage-qualified surface-water response


In [ ]:
jrc_rows = []

for event_id in event_ids:

    g = jrc_links.loc[
        jrc_links['event_id'] == event_id
    ].sort_values('native_month_mid').copy()

    if len(g) == 0:
        jrc_rows.append({
            'event_id': event_id,
            'JRC_response_metric_available_flag': 0
        })
        continue

    onset = g['onset_start_date'].iloc[0]
    event_end = g['event_end_date'].iloc[0]
    context_end = g['context_window_end'].iloc[0]

    usable = g.loc[
        g['water_fraction_z'].notna()
    ].copy()

    pre = usable.loc[
        usable['native_month_mid'] < onset
    ]

    pre_baseline = (
        float(pre['water_fraction_z'].median())
        if len(pre)
        else np.nan
    )

    event_usable = usable.loc[
        usable['overlap_event_days'] > 0
    ]

    event_z = weighted_mean(
        event_usable['water_fraction_z'],
        event_usable['overlap_event_days']
    )

    candidate = usable.loc[
        (usable['native_month_mid'] >= onset)
        &
        (usable['native_month_mid'] <= context_end)
    ].copy()

    if len(candidate) == 0:
        jrc_rows.append({
            'event_id': event_id,
            'JRC_response_metric_available_flag': 0,
            'prebaseline_JRC_records': len(pre),
            'prebaseline_water_fraction_z': pre_baseline
        })
        continue

    min_idx = candidate[
        'water_fraction_z'
    ].idxmin()

    min_z = candidate.loc[
        min_idx,
        'water_fraction_z'
    ]

    min_dev = candidate.loc[
        min_idx,
        'water_fraction_deviation_from_monthly_median'
    ]

    min_date = candidate.loc[
        min_idx,
        'native_month_mid'
    ]

    to_min = candidate.loc[
        candidate['native_month_mid'] <= min_date
    ]

    confounded = int(
        to_min[
            'overlaps_other_primary_event_flag'
        ].max()
    )

    normal_date, baseline_date = recovery_dates(
        usable,
        'native_month_mid',
        'water_fraction_z',
        min_date,
        context_end,
        pre_baseline
    )

    jrc_rows.append({
        'event_id': event_id,
        'primary_driver_class':
            g['primary_driver_class'].iloc[0],
        'onset_season':
            g['onset_season'].iloc[0],

        'JRC_response_metric_available_flag': 1,

        'prebaseline_JRC_records':
            len(pre),

        'prebaseline_water_fraction_z':
            pre_baseline,

        'event_weighted_water_fraction_z':
            event_z,

        'minimum_water_fraction_z_onset_to_post180':
            min_z,

        'minimum_water_fraction_drop_from_prebaseline_z':
            (
                min_z - pre_baseline
                if np.isfinite(pre_baseline)
                else np.nan
            ),

        'minimum_water_fraction_deviation_from_monthly_median':
            min_dev,

        'minimum_JRC_native_month_mid':
            min_date,

        'lag_onset_to_minimum_water_fraction_days':
            int((min_date - onset).days),

        'lag_event_end_to_minimum_water_fraction_days':
            int((min_date - event_end).days),

        'other_flash_drought_before_minimum_JRC_flag':
            confounded,

        'recovery_to_seasonal_normal_date':
            normal_date,

        'recovery_to_seasonal_normal_days_after_event_end':
            (
                (normal_date - event_end).days
                if pd.notna(normal_date)
                else np.nan
            ),

        'recovery_to_prebaseline_date':
            baseline_date,

        'recovery_to_prebaseline_days_after_event_end':
            (
                (baseline_date - event_end).days
                if pd.notna(baseline_date)
                else np.nan
            ),

        'seasonal_normal_recovery_censored_flag':
            int(pd.isna(normal_date)),

        'candidate_qualified_JRC_records_onset_to_post180':
            len(candidate)
    })


jrc_metrics = pd.DataFrame(jrc_rows)
jrc_metrics = jrc_metrics.rename(columns={'recovery_to_seasonal_normal_date': 'JRC_recovery_to_seasonal_normal_date', 'recovery_to_seasonal_normal_days_after_event_end': 'JRC_recovery_to_seasonal_normal_days_after_event_end', 'recovery_to_prebaseline_date': 'JRC_recovery_to_prebaseline_date', 'recovery_to_prebaseline_days_after_event_end': 'JRC_recovery_to_prebaseline_days_after_event_end', 'seasonal_normal_recovery_censored_flag': 'JRC_seasonal_normal_recovery_censored_flag'})

assert len(jrc_metrics) == 39
assert jrc_metrics['event_id'].is_unique

display(jrc_metrics.head())


## 8. 39-event cross-response synthesis

Recovery fields are branch-prefixed (`MOD16_`, `MOD13_`, `GRACE_`, `JRC_`) before merging so Pandas cannot create ambiguous duplicate recovery columns.


In [ ]:
event_synthesis = availability[[
    'event_id',
    'onset_pentad',
    'onset_start_date',
    'event_end_date',
    'total_duration_pentads',
    'prolonged_after_flash_flag',
    'primary_driver_class',
    'onset_season'
]].copy()

event_synthesis = (
    event_synthesis
    .merge(
        mod16_metrics.drop(
            columns=[
                'primary_driver_class',
                'onset_season'
            ]
        ),
        on='event_id',
        how='left',
        validate='one_to_one'
    )
    .merge(
        mod13_metrics.drop(
            columns=[
                'primary_driver_class',
                'onset_season'
            ]
        ),
        on='event_id',
        how='left',
        validate='one_to_one'
    )
    .merge(
        grace_metrics.drop(
            columns=[
                c for c in [
                    'primary_driver_class',
                    'onset_season'
                ]
                if c in grace_metrics.columns
            ]
        ),
        on='event_id',
        how='left',
        validate='one_to_one'
    )
    .merge(
        jrc_metrics.drop(
            columns=[
                c for c in [
                    'primary_driver_class',
                    'onset_season'
                ]
                if c in jrc_metrics.columns
            ]
        ),
        on='event_id',
        how='left',
        validate='one_to_one'
    )
)

assert len(event_synthesis) == 39
assert event_synthesis['event_id'].is_unique
assert not event_synthesis.columns.duplicated().any()

display(event_synthesis.head())


## 9. QC summary


In [ ]:
qc_rows = []


def add_qc(check, value, status, note):
    qc_rows.append({
        'check': check,
        'value': value,
        'status': status,
        'note': note
    })


add_qc(
    'Primary events retained',
    len(event_synthesis),
    'PASS',
    'Must remain 39.'
)

add_qc(
    'MOD16 event metrics',
    len(mod16_metrics),
    'PASS',
    'Expected 39.'
)

add_qc(
    'MOD13 event metrics',
    len(mod13_metrics),
    'PASS',
    'Expected 39.'
)

add_qc(
    'GRACE metric rows',
    len(grace_metrics),
    'PASS',
    '39 rows including unavailable events.'
)

add_qc(
    'JRC metric rows',
    len(jrc_metrics),
    'PASS',
    '39 rows including unavailable events.'
)

add_qc(
    'MOD16 missing prebaseline',
    int(
        mod16_metrics[
            'prebaseline_ETPET_z'
        ].isna().sum()
    ),
    'PASS',
    'Expected 0.'
)

add_qc(
    'MOD13 missing prebaseline',
    int(
        mod13_metrics[
            'prebaseline_NDVI_z'
        ].isna().sum()
    ),
    'PASS',
    'Expected 0.'
)

add_qc(
    'MOD16 seasonal-normal recovery censored',
    int(
        mod16_metrics[
            'MOD16_seasonal_normal_recovery_censored_flag'
        ].sum()
    ),
    'RESULT',
    'No interpolation/extrapolation.'
)

add_qc(
    'MOD13 seasonal-normal recovery censored',
    int(
        mod13_metrics[
            'MOD13_seasonal_normal_recovery_censored_flag'
        ].sum()
    ),
    'RESULT',
    'No interpolation/extrapolation.'
)

add_qc(
    'GRACE response metrics available',
    int(
        grace_metrics[
            'GRACE_response_metric_available_flag'
        ].fillna(0).sum()
    ),
    'RESULT',
    'Source-era / context support.'
)

add_qc(
    'JRC response metrics available',
    int(
        jrc_metrics[
            'JRC_response_metric_available_flag'
        ].fillna(0).sum()
    ),
    'RESULT',
    'Coverage-qualified climatology support.'
)

add_qc(
    'GRACE confounded-to-minimum metrics',
    int(
        grace_metrics[
            'other_flash_drought_before_minimum_TWSA_flag'
        ].fillna(0).sum()
    ),
    'RESULT',
    'Retain; do not treat as isolated response.'
)

add_qc(
    'JRC confounded-to-minimum metrics',
    int(
        jrc_metrics[
            'other_flash_drought_before_minimum_JRC_flag'
        ].fillna(0).sum()
    ),
    'RESULT',
    'Retain; do not treat as isolated response.'
)

add_qc(
    'JRC July qualified climatology n',
    int(
        jrc_month_stats.loc[7, 'JRC_clim_n']
    ) if 7 in jrc_month_stats.index else 0,
    'PASS',
    'Expected 0; July primary standardized JRC response remains unavailable.'
)

qc = pd.DataFrame(qc_rows)

display(qc)

assert len(event_synthesis) == 39
assert len(mod16_metrics) == 39
assert len(mod13_metrics) == 39
assert len(grace_metrics) == 39
assert len(jrc_metrics) == 39
assert mod16_metrics['prebaseline_ETPET_z'].notna().all()
assert mod13_metrics['prebaseline_NDVI_z'].notna().all()


## 10. Save tables


In [ ]:
MOD16_LINK_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_MOD16_Seasonal_Anomaly_Links.csv'
MOD13_LINK_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_MOD13_Seasonal_Anomaly_Links.csv'
GRACE_LINK_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_GRACE_TWSA_Seasonal_Anomaly_Links.csv'
JRC_LINK_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_JRC_Qualified_Seasonal_Anomaly_Links.csv'

MOD16_METRIC_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_MOD16_Event_Response_Metrics.csv'
MOD13_METRIC_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_MOD13_Event_Response_Metrics.csv'
GRACE_METRIC_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_GRACE_TWSA_Event_Response_Metrics.csv'
JRC_METRIC_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_JRC_SurfaceWater_Event_Response_Metrics.csv'

SYNTHESIS_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_Cross_Response_Event_Synthesis.csv'
JRC_CLIM_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_JRC_Qualified_Monthly_Climatology.csv'
QC_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_QC.csv'
CONFIG_OUT = PHASE2F_DIR / 'AEZ26_Phase2F_Method_Config.txt'

mod16_links.to_csv(MOD16_LINK_OUT, index=False, date_format='%Y-%m-%d')
mod13_links.to_csv(MOD13_LINK_OUT, index=False, date_format='%Y-%m-%d')
grace_links.to_csv(GRACE_LINK_OUT, index=False, date_format='%Y-%m-%d')
jrc_links.to_csv(JRC_LINK_OUT, index=False, date_format='%Y-%m-%d')

mod16_metrics.to_csv(MOD16_METRIC_OUT, index=False, date_format='%Y-%m-%d')
mod13_metrics.to_csv(MOD13_METRIC_OUT, index=False, date_format='%Y-%m-%d')
grace_metrics.to_csv(GRACE_METRIC_OUT, index=False, date_format='%Y-%m-%d')
jrc_metrics.to_csv(JRC_METRIC_OUT, index=False, date_format='%Y-%m-%d')

event_synthesis.to_csv(SYNTHESIS_OUT, index=False, date_format='%Y-%m-%d')
jrc_month_stats.to_csv(JRC_CLIM_OUT)
qc.to_csv(QC_OUT, index=False)

method_config = '''PHASE 2F METHOD CONFIG
Flash-drought catalogue changed: NO

MOD16 climatology:
- fixed 8-day seasonal slot
- primary response = standardized ET/PET ratio
- ET and PET standardized separately as diagnostics
- strict-QA ET/PET retained as sensitivity
- MOD16 ET/PET z is NOT called ESI

MOD13 climatology:
- fixed 16-day seasonal slot
- primary = standard-QA NDVI/EVI standardized anomaly
- event/pre means weighted by standard_QA_fraction
- operational sensitivities: standard_QA_fraction >=0.25 and >=0.50
- strict-QA retained as sensitivity

GRACE:
- calendar-month TWSA climatology
- no interpolation
- TWSA is NOT direct groundwater
- event-confounding flag retained

JRC:
- primary response = water_fraction_of_valid_area
- climatology uses only coverage-qualified observations
- minimum monthly climatology n = 5
- July has no qualified climatology and remains missing
- water fraction is NOT storage volume

Recovery:
- first native observation after minimum with z >= 0
- first native observation after minimum with z >= pre-event baseline
- not observed within context = censored, no extrapolation

No formal causal inference or final ML fitting in Phase 2F.
'''

CONFIG_OUT.write_text(method_config, encoding='utf-8')

print('Saved Phase 2F tables.')


## 11. Diagnostic figures


In [ ]:
# MOD16 response depth
plt.figure(figsize=(8,5))
plt.hist(
    mod16_metrics[
        'minimum_ETPET_drop_from_prebaseline_z'
    ].dropna()
)
plt.xlabel('Minimum ET/PET z change from pre-event baseline')
plt.ylabel('Number of events')
plt.title('AEZ-26 MOD16 Evaporative-Stress Response Depth')
plt.tight_layout()
plt.savefig(
    FIG_DIR / 'AEZ26_Phase2F_MOD16_ETPET_Response_Depth.png',
    dpi=200,
    bbox_inches='tight'
)
plt.show()


In [ ]:
# MOD13 NDVI response lag
plt.figure(figsize=(8,5))
plt.hist(
    mod13_metrics[
        'lag_onset_to_minimum_NDVI_days'
    ].dropna()
)
plt.xlabel('Days from flash-drought onset to minimum NDVI anomaly')
plt.ylabel('Number of events')
plt.title('AEZ-26 MOD13 NDVI Response Lag')
plt.tight_layout()
plt.savefig(
    FIG_DIR / 'AEZ26_Phase2F_MOD13_NDVI_Response_Lag.png',
    dpi=200,
    bbox_inches='tight'
)
plt.show()


In [ ]:
# GRACE TWSA lag for available metrics
plot_grace = grace_metrics.loc[
    grace_metrics[
        'GRACE_response_metric_available_flag'
    ] == 1
]

plt.figure(figsize=(8,5))
plt.hist(
    plot_grace[
        'lag_onset_to_minimum_TWSA_days'
    ].dropna()
)
plt.xlabel('Days from flash-drought onset to minimum TWSA anomaly')
plt.ylabel('Number of events')
plt.title('AEZ-26 GRACE TWSA Response Lag')
plt.tight_layout()
plt.savefig(
    FIG_DIR / 'AEZ26_Phase2F_GRACE_TWSA_Response_Lag.png',
    dpi=200,
    bbox_inches='tight'
)
plt.show()


In [ ]:
# Cross-response depth scatter: MOD16 vs NDVI
plot_cross = event_synthesis.dropna(
    subset=[
        'minimum_ETPET_drop_from_prebaseline_z',
        'minimum_NDVI_drop_from_prebaseline_z'
    ]
)

plt.figure(figsize=(7,6))
plt.scatter(
    plot_cross[
        'minimum_ETPET_drop_from_prebaseline_z'
    ],
    plot_cross[
        'minimum_NDVI_drop_from_prebaseline_z'
    ]
)
plt.axhline(0)
plt.axvline(0)
plt.xlabel('MOD16 minimum ET/PET change from pre-baseline (z)')
plt.ylabel('MOD13 minimum NDVI change from pre-baseline (z)')
plt.title('AEZ-26 Functional vs Structural Response Depth')
plt.tight_layout()
plt.savefig(
    FIG_DIR / 'AEZ26_Phase2F_MOD16_vs_MOD13_Response_Depth.png',
    dpi=200,
    bbox_inches='tight'
)
plt.show()


## 12. Create ZIP audit package


In [ ]:
ZIP_BASE = PHASE2_DIR / 'AEZ26_Phase2F_Output_Package'

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    'zip',
    root_dir=PHASE2_DIR,
    base_dir='Phase2F'
)

print('ZIP CREATED:')
print(zip_path)


In [ ]:
print('FINAL PHASE 2F CHECK')
print('Events:', len(event_synthesis))
print('MOD16 prebaseline missing:',
      mod16_metrics['prebaseline_ETPET_z'].isna().sum())
print('MOD13 prebaseline missing:',
      mod13_metrics['prebaseline_NDVI_z'].isna().sum())
print('MOD16 seasonal-normal recovery censored:',
      mod16_metrics['MOD16_seasonal_normal_recovery_censored_flag'].sum())
print('MOD13 seasonal-normal recovery censored:',
      mod13_metrics['MOD13_seasonal_normal_recovery_censored_flag'].sum())
print('GRACE metric available:',
      int(grace_metrics['GRACE_response_metric_available_flag'].fillna(0).sum()))
print('JRC metric available:',
      int(jrc_metrics['JRC_response_metric_available_flag'].fillna(0).sum()))
print()
print('Upload this ZIP for independent audit:')
print(zip_path)


## Next gate

Upload the Phase 2F ZIP for independent audit.

After Phase 2F passes, the next stage will combine:

- driver-response statistical synthesis,
- clean vs confounded lag sensitivity,
- groundwater-well integration when BWDB data arrive,
- and then temporal ML / physics-informed modeling.

Do not substitute GRACE TWSA for observed groundwater.
